# Bot Workflow Runs Summary Extractor

**Date:** 2026-02-08  
**Purpose:** Extract key information from GitHub Actions workflow runs and create Excel summary

**Project Documentation:** `conversation and context docs/Extract Run Info from GitHub Actions Implementation Plan 02-08-2026.md`

This notebook:
1. Fetches workflow runs from GitHub Actions API
2. Downloads logs for each run
3. Extracts: run number, timestamp, question presence, question number, forecast value, error flag
4. Outputs to Excel file: `products/Runs and Question Numbers.xlsx`

---

## Setup & Configuration

In [1]:
# Install openpyxl if needed
# !pip install openpyxl

In [2]:
import json
import re
import subprocess
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional
from dataclasses import dataclass, asdict

from openpyxl import load_workbook, Workbook
from openpyxl.styles import Font, Alignment

print("✅ Imports successful")

✅ Imports successful


In [3]:
# Configuration
REPO = "D-Enns/metac-bot-template"
WORKFLOW = "dre_run_bot_on_tournament.yaml"
OUTPUT_FILE = Path(r"C:/Users/Donni/projects/metac_bot_Spring_2026/products/Runs and Question Numbers.xlsx")

# For testing - limit number of runs to process
TEST_LIMIT = 10  # Set to None to process all runs

print(f"Repository: {REPO}")
print(f"Workflow: {WORKFLOW}")
print(f"Output: {OUTPUT_FILE}")
print(f"Test limit: {TEST_LIMIT}")

Repository: D-Enns/metac-bot-template
Workflow: dre_run_bot_on_tournament.yaml
Output: C:\Users\Donni\projects\metac_bot_Spring_2026\products\Runs and Question Numbers.xlsx
Test limit: 10


In [4]:
# Data structure for extracted run information
@dataclass
class RunInfo:
    workflow_run_number: int
    time_date: str
    has_question: str  # "Y" or "N"
    question_number: str  # Empty string if no question
    forecast_value: str  # JSON string or single value
    error_flag: str  # "Y" or "N"

print("✅ Data structure defined")

✅ Data structure defined


## GitHub CLI Functions

In [5]:
def verify_gh_cli() -> bool:
    """Verify gh CLI is installed and authenticated."""
    try:
        result = subprocess.run(
            ["gh", "auth", "status"],
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode != 0:
            print("❌ Error: gh CLI is not authenticated. Run 'gh auth login' first.")
            return False
        print("✅ GitHub CLI authenticated")
        return True
    except FileNotFoundError:
        print("❌ Error: gh CLI is not installed. Install from https://cli.github.com/")
        return False
    except Exception as e:
        print(f"❌ Error checking gh CLI: {e}")
        return False

# Test authentication
verify_gh_cli()

✅ GitHub CLI authenticated


True

In [16]:
# test cell
# Debug: See raw output from gh api
cmd = [
    "gh", "api",
    f"repos/{REPO}/actions/workflows/{WORKFLOW}/runs"
]

result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)

print(f"Return code: {result.returncode}")
print(f"\nStderr: {result.stderr}")
print(f"\nStdout (first 2000 chars):")
print("=" * 80)
print(result.stdout[:2000])
print("=" * 80)

# Try to parse it
if result.stdout:
    try:
        data = json.loads(result.stdout)
        print(f"\n✅ Valid JSON!")
        print(f"Keys in response: {data.keys()}")
        if 'workflow_runs' in data:
            print(f"Number of workflow_runs: {len(data['workflow_runs'])}")
            if data['workflow_runs']:
                print(f"\nFirst run sample:")
                print(json.dumps(data['workflow_runs'][0], indent=2)[:500])
    except Exception as e:
        print(f"\n❌ JSON parsing failed: {e}")

Return code: 0

Stderr: 

Stdout (first 2000 chars):
{
  "total_count": 1224,
  "workflow_runs": [
    {
      "id": 21806822182,
      "name": "Dre Tournament Bot",
      "node_id": "WFR_kwLOOh99KM8AAAAFE8mzJg",
      "head_branch": "bot-dev",
      "head_sha": "91556ccc0f7da5dc1c0d7d9d82382b2e5ac7aea0",
      "path": ".github/workflows/dre_run_bot_on_tournament.yaml",
      "display_title": "Dre Tournament Bot",
      "run_number": 1224,
      "event": "schedule",
      "status": "completed",
      "conclusion": "success",
      "workflow_id": 220638909,
      "check_suite_id": 56788368829,
      "check_suite_node_id": "CS_kwDOOh99KM8AAAANONm9vQ",
      "url": "https://api.github.com/repos/D-Enns/metac-bot-template/actions/runs/21806822182",
      "html_url": "https://github.com/D-Enns/metac-bot-template/actions/runs/21806822182",
      "pull_requests": [],
      "created_at": "2026-02-08T22:48:31Z",
      "updated_at": "2026-02-08T22:49:29Z",
      "actor": {
        "login": "D-Enn

In [17]:
# Simpler test - parse just the first response (no pagination)
cmd = [
    "gh", "api",
    f"repos/{REPO}/actions/workflows/{WORKFLOW}/runs"
]

result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)

print(f"Output length: {len(result.stdout)}")
print(f"First char: {repr(result.stdout[0]) if result.stdout else 'EMPTY'}")
print(f"Output starts with bracket: {result.stdout.startswith('{')}")

# Try parsing
try:
    data = json.loads(result.stdout)
    print(f"\n✅ Parsed successfully!")
    print(f"Total runs available: {data['total_count']}")
    print(f"Runs in this page: {len(data['workflow_runs'])}")

    # Show first run
    if data['workflow_runs']:
        first_run = data['workflow_runs'][0]
        print(f"\nFirst run:")
        print(f"  Run #: {first_run['run_number']}")
        print(f"  ID: {first_run['id']}")
        print(f"  Created: {first_run['created_at']}")
        print(f"  Status: {first_run['conclusion']}")
except json.JSONDecodeError as e:
    print(f"\n❌ JSON parsing failed: {e}")
    print(f"\nFirst 100 chars of stdout:")
    print(repr(result.stdout[:100]))

Output length: 686481
First char: '\x1b'
Output starts with bracket: False

❌ JSON parsing failed: Expecting value: line 1 column 1 (char 0)

First 100 chars of stdout:
'\x1b{\x1b[m\n  \x1b"total_count"\x1b[m\x1b:\x1b[m 1224\x1b,\x1b[m\n  \x1b"workflow_runs"\x1b[m\x1b:'


In [15]:
def get_workflow_runs(repo: str, workflow: str, limit: Optional[int] = None) -> List[Dict]:
      """Fetch workflow runs from GitHub Actions API."""
      print(f"Fetching workflow runs from {repo}...")

      # Get raw JSON without jq formatting
      cmd = [
          "gh", "api",
          f"repos/{repo}/actions/workflows/{workflow}/runs",
          "--paginate"
      ]

      try:
          result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)

          # Check for errors
          if result.returncode != 0:
              print(f"❌ Command failed with return code {result.returncode}")
              print(f"Error output: {result.stderr}")
              return []

          # Check if output is empty
          if not result.stdout.strip():
              print(f"❌ Command returned empty output")
              return []

          # Parse the JSON response
          # With --paginate, we get multiple JSON objects concatenated
          # Each is a complete response with workflow_runs array
          runs = []

          # Split by '}' followed by '{' to separate paginated responses
          json_chunks = result.stdout.strip()

          # Try to parse as single JSON first
          try:
              data = json.loads(json_chunks)
              if 'workflow_runs' in data:
                  runs.extend(data['workflow_runs'])
          except json.JSONDecodeError:
              # If that fails, try splitting into multiple JSON objects
              # This happens with pagination
              import re
              # Find all complete JSON objects
              objects = re.findall(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', json_chunks, re.DOTALL)
              for obj_str in objects:
                  try:
                      data = json.loads(obj_str)
                      if 'workflow_runs' in data:
                          runs.extend(data['workflow_runs'])
                  except json.JSONDecodeError:
                      continue

          if not runs:
              print("❌ No runs found in response")
              return []

          # Extract just the fields we need
          simplified_runs = []
          for run in runs:
              simplified_runs.append({
                  'id': run['id'],
                  'run_number': run['run_number'],
                  'created_at': run['created_at'],
                  'conclusion': run.get('conclusion', '')
              })

          # Sort by run_number descending (newest first)
          simplified_runs.sort(key=lambda x: x['run_number'], reverse=True)

          # Apply limit if specified
          if limit:
              simplified_runs = simplified_runs[:limit]

          print(f"✅ Found {len(simplified_runs)} runs")
          return simplified_runs

      except Exception as e:
          print(f"❌ Error fetching runs: {e}")
          import traceback
          traceback.print_exc()
          return []

# Test: Fetch limited runs
test_runs = get_workflow_runs(REPO, WORKFLOW, limit=5)
if test_runs:
    print("\nSample run:")
    print(f"  Run #{test_runs[0]['run_number']}")
    print(f"  ID: {test_runs[0]['id']}")
    print(f"  Created: {test_runs[0]['created_at']}")
    print(f"  Conclusion: {test_runs[0]['conclusion']}")

Fetching workflow runs from D-Enns/metac-bot-template...
❌ No runs found in response


In [ ]:
def download_run_log(repo: str, run_id: int, run_number: int) -> Optional[str]:
    """Download log for a single workflow run."""
    cmd = ["gh", "run", "view", str(run_id), "--repo", repo, "--log"]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        
        if result.returncode != 0:
            print(f"  ❌ Failed to download run {run_number}")
            return None
        
        return result.stdout
        
    except Exception as e:
        print(f"  ❌ Error downloading run {run_number}: {e}")
        return None

print("✅ Download function defined")

## Test: Download and Inspect Single Log

Let's download one log to see what we're working with.

In [ ]:
# Get the most recent run
if test_runs:
    sample_run = test_runs[0]
    print(f"Downloading log for run #{sample_run['run_number']}...")
    sample_log = download_run_log(REPO, sample_run['id'], sample_run['run_number'])
    
    if sample_log:
        print(f"✅ Downloaded log ({len(sample_log)} characters)")
        print(f"\nFirst 1000 characters:")
        print("=" * 80)
        print(sample_log[:1000])
        print("=" * 80)
    else:
        print("❌ Failed to download log")

## Data Extraction Functions

Now let's build the functions to extract each piece of information from the logs.

In [ ]:
# Regex patterns (reused from existing tools)
TIMESTAMP_PATTERN = re.compile(r'(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+Z)')
QUESTION_URL_PATTERN = re.compile(r'https://www\.metaculus\.com/questions/(\d+)')
FOUND_RESEARCH_PATTERN = re.compile(r'Found Research for URL\s+(https://www\.metaculus\.com/questions/\d+)')
ERROR_EXIT_PATTERN = re.compile(r'Error: Process completed with exit code 1\.')

print("✅ Regex patterns defined")

In [ ]:
def extract_timestamp(log_content: str) -> str:
    """Extract first timestamp from 'Run bot' section."""
    # Find the line with "##[group]Run poetry run python main.py"
    for line in log_content.split('\n')[:50]:  # Check first 50 lines
        if "##[group]Run poetry run python main.py" in line or "Run poetry run python main.py" in line:
            match = TIMESTAMP_PATTERN.search(line)
            if match:
                timestamp_str = match.group(1)
                # Convert: "2026-01-03T14:02:21.9983260Z" -> "2026-01-03 14:02:21"
                dt = datetime.fromisoformat(timestamp_str.replace('Z', '+00:00'))
                return dt.strftime('%Y-%m-%d %H:%M:%S')
    
    return ""  # Fallback if not found

print("✅ extract_timestamp() defined")

In [ ]:
def check_has_question(log_content: str) -> str:
    """Check if run processed a question (Y/N)."""
    if FOUND_RESEARCH_PATTERN.search(log_content):
        return "Y"
    return "N"

print("✅ check_has_question() defined")

In [ ]:
def extract_question_number(log_content: str) -> str:
    """Extract question number from Metaculus URL."""
    match = QUESTION_URL_PATTERN.search(log_content)
    return match.group(1) if match else ""

print("✅ extract_question_number() defined")

In [ ]:
def detect_question_type(log_content: str) -> Optional[str]:
    """Detect question type from log content."""
    if "BinaryQuestion" in log_content or "*Final Prediction*:" in log_content:
        return "Binary"
    elif "MultipleChoiceQuestion" in log_content:
        return "MultipleChoice"
    elif "NumericQuestion" in log_content or "Probability distribution:" in log_content:
        return "Numeric"
    return None

print("✅ detect_question_type() defined")

In [ ]:
def extract_binary_forecast(log_content: str) -> str:
    """Extract binary forecast percentage."""
    patterns = [
        r'\*Final Prediction\*:\s*(\d+\.?\d*)%?',
        r'\*\*Probability:\s*(\d+\.?\d*)%?'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, log_content)
        if match:
            return match.group(1)
    
    raise ValueError("Binary forecast not found")

print("✅ extract_binary_forecast() defined")

In [ ]:
def extract_mc_forecast(log_content: str) -> str:
    """Extract multiple choice forecast as JSON."""
    # Look for "# Final Answer" section with JSON dict
    section_match = re.search(
        r'# Final Answer\s*\n\s*(\{[^}]+\})',
        log_content
    )
    
    if section_match:
        return section_match.group(1)
    
    # Alternative: Parse bullet list format
    # - Option A: 60.5%
    # - Option B: 39.5%
    pattern = r'-\s*([^:]+):\s*(\d+\.?\d*)%?'
    matches = re.findall(pattern, log_content)
    
    if matches:
        forecast_dict = {opt.strip(): float(pct) for opt, pct in matches}
        return json.dumps(forecast_dict)
    
    raise ValueError("Multiple choice forecast not found")

print("✅ extract_mc_forecast() defined")

In [ ]:
def extract_numeric_forecast(log_content: str) -> str:
    """Extract numeric forecast list as JSON."""
    # Look for "# Final Answer" followed by array
    section_match = re.search(
        r'# Final Answer\s*\n\s*(\[[\d.,\s]+\])',
        log_content
    )
    
    if section_match:
        return section_match.group(1)
    
    raise ValueError("Numeric forecast list not found")

print("✅ extract_numeric_forecast() defined")

In [ ]:
def extract_forecast_value(log_content: str, question_type: Optional[str]) -> str:
    """Extract forecast value based on question type."""
    if not question_type:
        return ""
    
    try:
        if question_type == "Binary":
            return extract_binary_forecast(log_content)
        elif question_type == "MultipleChoice":
            return extract_mc_forecast(log_content)
        elif question_type == "Numeric":
            return extract_numeric_forecast(log_content)
    except Exception as e:
        return f"ERROR: {str(e)}"
    
    return ""

print("✅ extract_forecast_value() defined")

In [ ]:
def check_error_flag(log_content: str) -> str:
    """Check if run had error exit code (Y/N)."""
    # Check last 200 lines for error pattern
    last_lines = '\n'.join(log_content.split('\n')[-200:])
    
    if ERROR_EXIT_PATTERN.search(last_lines):
        return "Y"
    return "N"

print("✅ check_error_flag() defined")

In [ ]:
def process_log(log_content: str, run_number: int) -> RunInfo:
    """Process a single log file and extract all fields."""
    
    # Extract timestamp
    time_date = extract_timestamp(log_content)
    
    # Check for question
    has_question = check_has_question(log_content)
    
    # Extract question number (if present)
    question_number = extract_question_number(log_content) if has_question == "Y" else ""
    
    # Extract forecast value (if question present)
    forecast_value = ""
    if has_question == "Y":
        question_type = detect_question_type(log_content)
        forecast_value = extract_forecast_value(log_content, question_type)
    
    # Check error flag
    error_flag = check_error_flag(log_content)
    
    return RunInfo(
        workflow_run_number=run_number,
        time_date=time_date,
        has_question=has_question,
        question_number=question_number,
        forecast_value=forecast_value,
        error_flag=error_flag
    )

print("✅ process_log() defined")

## Test: Extract Data from Sample Log

Let's test our extraction functions on the sample log we downloaded.

In [ ]:
if sample_log:
    print(f"Testing extraction on run #{sample_run['run_number']}...\n")
    
    # Test each extraction function
    print(f"Timestamp: {extract_timestamp(sample_log)}")
    print(f"Has question: {check_has_question(sample_log)}")
    print(f"Question number: {extract_question_number(sample_log)}")
    print(f"Question type: {detect_question_type(sample_log)}")
    print(f"Error flag: {check_error_flag(sample_log)}")
    
    # Test full processing
    print("\n" + "=" * 80)
    print("Full RunInfo extraction:")
    print("=" * 80)
    run_info = process_log(sample_log, sample_run['run_number'])
    print(f"Run Number: {run_info.workflow_run_number}")
    print(f"Time/Date: {run_info.time_date}")
    print(f"Has Question: {run_info.has_question}")
    print(f"Question Number: {run_info.question_number}")
    print(f"Forecast Value: {run_info.forecast_value}")
    print(f"Error Flag: {run_info.error_flag}")
else:
    print("⚠️  No sample log available for testing")

## Process Multiple Runs

Now let's process multiple runs to build our dataset.

In [ ]:
def process_runs(repo: str, workflow: str, limit: Optional[int] = None) -> List[RunInfo]:
    """Process multiple workflow runs and extract data."""
    
    # Get runs
    runs = get_workflow_runs(repo, workflow, limit=limit)
    
    if not runs:
        print("No runs found")
        return []
    
    print(f"\nProcessing {len(runs)} runs...\n")
    
    run_info_list = []
    
    for i, run in enumerate(runs, 1):
        run_id = run['id']
        run_number = run['run_number']
        
        print(f"[{i}/{len(runs)}] Processing run #{run_number}...", end='')
        
        # Download log
        log_content = download_run_log(repo, run_id, run_number)
        
        if not log_content:
            print(" ❌ Failed to download")
            continue
        
        # Extract info
        try:
            run_info = process_log(log_content, run_number)
            run_info_list.append(run_info)
            print(f" ✓ (Question: {run_info.has_question}, Error: {run_info.error_flag})")
        except Exception as e:
            print(f" ⚠️  Error: {e}")
    
    print(f"\n✅ Processed {len(run_info_list)} runs successfully")
    return run_info_list

print("✅ process_runs() defined")

In [ ]:
# Process test batch
print(f"Processing {TEST_LIMIT} runs for testing...\n")
test_data = process_runs(REPO, WORKFLOW, limit=TEST_LIMIT)

In [ ]:
# Inspect the data
if test_data:
    print("\n" + "=" * 80)
    print("Sample Extracted Data:")
    print("=" * 80)
    for info in test_data[:3]:  # Show first 3
        print(f"\nRun #{info.workflow_run_number}:")
        print(f"  Time: {info.time_date}")
        print(f"  Has Question: {info.has_question}")
        print(f"  Question #: {info.question_number}")
        print(f"  Forecast: {info.forecast_value[:100] if len(info.forecast_value) > 100 else info.forecast_value}")
        print(f"  Error: {info.error_flag}")

## Excel Output Functions

In [ ]:
def write_to_excel(run_info_list: List[RunInfo], output_file: Path):
    """Write extracted data to Excel file."""
    
    # Create or load workbook
    if output_file.exists():
        print(f"Loading existing workbook: {output_file}")
        wb = load_workbook(output_file)
        ws = wb.active
    else:
        print(f"Creating new workbook: {output_file}")
        wb = Workbook()
        ws = wb.active
        
        # Create header row
        headers = [
            'workflow_run_number',
            'time_date',
            'has_question',
            'question_number',
            'forecast_value',
            'error_flag'
        ]
        ws.append(headers)
        
        # Format header row
        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.alignment = Alignment(horizontal='center')
    
    # Sort by run_number descending
    sorted_runs = sorted(run_info_list, key=lambda x: x.workflow_run_number, reverse=True)
    
    # Append data rows
    for run_info in sorted_runs:
        ws.append([
            run_info.workflow_run_number,
            run_info.time_date,
            run_info.has_question,
            run_info.question_number,
            run_info.forecast_value,
            run_info.error_flag
        ])
    
    # Set column widths
    ws.column_dimensions['A'].width = 20  # workflow_run_number
    ws.column_dimensions['B'].width = 20  # time_date
    ws.column_dimensions['C'].width = 15  # has_question
    ws.column_dimensions['D'].width = 18  # question_number
    ws.column_dimensions['E'].width = 50  # forecast_value
    ws.column_dimensions['F'].width = 12  # error_flag
    
    # Ensure parent directory exists
    output_file.parent.mkdir(parents=True, exist_ok=True)
    
    # Save
    wb.save(output_file)
    print(f"✅ Excel file saved: {output_file}")
    print(f"   Total rows: {len(sorted_runs)} (plus header)")

print("✅ write_to_excel() defined")

In [ ]:
# Write test data to Excel
if test_data:
    write_to_excel(test_data, OUTPUT_FILE)
else:
    print("⚠️  No data to write")

## Full Processing

Once the test looks good, run this cell to process ALL runs (no limit).

In [ ]:
# UNCOMMENT TO PROCESS ALL RUNS (this will take a while!)
# print("⚠️  Processing ALL runs - this may take 30-60 minutes...\n")
# full_data = process_runs(REPO, WORKFLOW, limit=None)
# if full_data:
#     write_to_excel(full_data, OUTPUT_FILE)
#     print(f"\n🎉 Complete! Processed {len(full_data)} runs")

---

## Summary Statistics

Optional: Analyze the extracted data.

In [ ]:
if test_data:
    total_runs = len(test_data)
    runs_with_questions = sum(1 for r in test_data if r.has_question == "Y")
    runs_with_errors = sum(1 for r in test_data if r.error_flag == "Y")
    
    print("=" * 80)
    print("SUMMARY STATISTICS")
    print("=" * 80)
    print(f"Total runs processed: {total_runs}")
    print(f"Runs with questions: {runs_with_questions} ({runs_with_questions/total_runs*100:.1f}%)")
    print(f"Runs with errors: {runs_with_errors} ({runs_with_errors/total_runs*100:.1f}%)")
    print(f"Success rate: {(total_runs - runs_with_errors)/total_runs*100:.1f}%")
    print("=" * 80)